# 3-Month Follow-Up Adherence & Resurgery Rate Trend Analysis

This notebook tracks **1-Month Follow-Up Adherence** and **1-Month Resurgery Rates** across **3 consecutive monthly datasets**.

### Clinical Definitions:
- **1-Month Follow-Up Adherence**: Evaluated strictly for visits occurring between **11 and 45 days** post-surgery (`11 <= days_after_surgery <= 45`). A primary surgery is Adherent (`YES`) if at least one visit occurred in this window.
- **1-Month Resurgeries**: Evaluated strictly for repeat procedures occurring between **11 and 45 days** post-surgery (`11 <= days_after_surgery <= 45`), categorized into **Rebubbling**, **Wound Resuturing**, **KP (Keratoplasty)**, and **Others**.
- **Trend Comparisons**: Side-by-side comparison across Month 1, Month 2, Month 3, 3-Month Pooled Total, and Trend Change (M3 vs M1).
- **Deliverables**: Formatted Excel Workbook (`Multi_Month_Trend_Results.xlsx`) and Executive Multi-Page PDF Charts Report (`Multi_Month_Trend_Report.pdf`).

In [ ]:
import os
import re
import io
import pandas as pd
import numpy as np
import openpyxl
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
from datetime import datetime

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

# ── Specify 3 Consecutive Monthly Files ──
FILE_MONTH_1 = r'data/Given Corn Tran Surg Px  Adv and FUP Data Details-Mar2026.xlsb'
FILE_MONTH_2 = r'data/Given Corn Tran Surg Px  Adv and FUP Data Details-April2026.xlsb'
FILE_MONTH_3 = r'data/Given Corn Tran Surg Px  Adv and FUP Data Details Nisha.xlsb'

OUTPUT_EXCEL = r'Multi_Month_Trend_Results.xlsx'
OUTPUT_PDF   = r'Multi_Month_Trend_Report.pdf'

print('✓ Configuration ready.')

In [ ]:
# ── Import Modular Pipeline Functions ──
from src.trend_analysis import (
    extract_month_label,
    load_and_clean_month_df,
    process_single_month,
    build_3month_trend_tables,
    create_trend_excel,
    create_trend_pdf,
)

print('✓ Pipeline module loaded.')

In [ ]:
# ── Extract Month Labels ──
input_files = [FILE_MONTH_1, FILE_MONTH_2, FILE_MONTH_3]
month_labels = [extract_month_label(f, idx + 1) for idx, f in enumerate(input_files)]

print('Detected Month Labels:')
for i, (m, f) in enumerate(zip(month_labels, input_files), 1):
    print(f'  Month {i}: {m}  <--  {os.path.basename(f)}')

# ── Process Each Monthly Dataset ──
month_results = {}
for label, fpath in zip(month_labels, input_files):
    print(f'\nLoading & processing {label}...')
    df_month = load_and_clean_month_df(fpath)
    res = process_single_month(df_month)
    month_results[label] = res
    n_surg = len(res['surgery_df'])
    n_adh = res['surgery_df']['is_adherent_1m'].sum()
    n_rep = len(res['repeat_1m'])
    print(f'  ✓ {label}: {n_surg} Primary Surgeries | 1M Adherent: {n_adh} ({n_adh/n_surg*100:.1f}%) | 1M Resurgeries: {n_rep} ({n_rep/n_surg*100:.2f}%)')

In [ ]:
# ── Build 6 Side-by-Side Trend Tables ──
trend_tables = build_3month_trend_tables(month_results)

print('✓ All 6 Trend Tables constructed successfully:')
for name in trend_tables:
    print(f'  • {name}')

In [ ]:
# ── Display Table 1: Adherence Trend Overall ──
print('=== 1. Adherence Trend Overall ===')
display(trend_tables['1_Trend_Adherence_Overall'])

In [ ]:
# ── Display Table 2: Adherence Trend by Campus ──
print('=== 2. Adherence Trend by Campus ===')
display(trend_tables['2_Trend_Adherence_Campus'])

In [ ]:
# ── Display Table 3: Adherence Trend by Surgery Procedure ──
print('=== 3. Adherence Trend by Surgery Procedure ===')
display(trend_tables['3_Trend_Adherence_SurgType'])

In [ ]:
# ── Display Table 4: Resurgery Trend Overall & Categories ──
print('=== 4. Resurgery Trend Overall & Categories ===')
display(trend_tables['4_Trend_Resurgery_Overall'])

In [ ]:
# ── Display Table 5 & 6: Resurgery Trend by Campus & Surgery ──
print('=== 5. Resurgery Trend by Campus ===')
display(trend_tables['5_Trend_Resurgery_Campus'])

print('=== 6. Resurgery Trend by Surgery Procedure ===')
display(trend_tables['6_Trend_Resurgery_SurgType'])

In [ ]:
# ── Export Executive Formatted Excel Workbook ──
excel_buf = create_trend_excel(trend_tables)
with open(OUTPUT_EXCEL, 'wb') as f:
    f.write(excel_buf.getvalue())

print(f'✓ Executive Excel Workbook saved to: {OUTPUT_EXCEL}')

In [ ]:
# ── Generate & Export Multi-Page PDF Charts Report ──
pdf_buf = create_trend_pdf(trend_tables, month_labels)
with open(OUTPUT_PDF, 'wb') as f:
    f.write(pdf_buf.getvalue())

print(f'✓ Executive Multi-Page PDF Charts Report saved to: {OUTPUT_PDF}')